In [1]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [ ]:
import os
from torch.utils.data import Dataset
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig
from tqdm import tqdm
import torch
import time

/data/miniconda/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!nvidia-smi

Sun Jul  5 21:36:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-32GB           Off |   00000000:07:00.0 Off |                    0 |
| N/A   32C    P0             44W /  300W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from pynvml import *
def print_gpu_utilization():
 nvmlInit()
 handle = nvmlDeviceGetHandleByIndex(0)
 info = nvmlDeviceGetMemoryInfo(handle)
 print(f"GPU memory occupied: {info.used//1024**2} MB.")

In [5]:
def load_dataset(filename):
    data_list = []
    with open(filename, "r", encoding="gb18030") as f:
        i = 0
        for line in f:
            i += 1
            if i < 10:
                print(line)
            try:
                dept, title, ques, ans = line.strip("\n").split(',', 4)
                data_list.append(
                    {
                        'department': dept,
                        'input': ques,
                        'output': ans
                    }
                )
            except:
                pass
    return data_list                  
                        
                        

In [6]:
data_list = load_dataset("./儿科5-14000.csv") 


department,title,ask,answer

营养保健科,小儿肥胖超重该如何治疗,女宝宝，刚7岁，这一年，察觉到，我家孩子身上肉很多，而且，食量非常的大，平时都不喜欢吃去玩，请问：小儿肥胖超重该如何治疗。,孩子出现肥胖症的情况。家长要通过孩子运功和健康的饮食来缓解他的症状，可以先让他做一些有氧运动，比如慢跑，爬坡，游泳等，并且饮食上孩子多吃黄瓜，胡萝卜，菠菜等，禁止孩子吃一些油炸食品和干果类食物，这些都是干热量高脂肪的食物，而且不要让孩子总是吃完就躺在床上不动，家长在治疗小儿肥胖期间如果孩子情况严重就要及时去医院在医生的指导下给孩子治疗。

营养保健科,小儿肥胖超重该怎样医治,男孩子，刚4岁，最近，发现，我家孩子体重要比别的孩子重很多，而且，最近越来越能吃了，还特别的懒，请问：小儿肥胖超重该怎样医治。,孩子一旦患上肥胖症家长要先通过运动和饮食来改变孩子的情况，要让孩子做一些他这个年龄段能做的运动，如游泳，慢跑等，要给孩子多吃一些像苹果，猕猴桃，胡萝卜等食物，禁止孩子吃高热量，高脂肪的食物，像蛋糕，干果，曲奇饼干等，严格的控制孩子的饮食，不要让他暴饮暴食，多运动对改变孩子肥胖都是有好处的，在治疗小儿肥胖期间如果情况严重，建议家长先带孩子去医院检查一下孩子肥胖症的原因在针对性的治疗。

营养保健科,小儿肥胖能吃该如何治疗,男宝，已经5岁，今年，察觉到，孩子身上越来越肉乎了，同时，吃的饭也比一般孩子多，平时都不喜欢吃去玩，请问：小儿肥胖能吃该如何治疗。,当孩子患上肥胖症的时候家长可以增加孩子的运动量和控制他的饮食来改变症状，像游泳，爬坡这类游泳运动对肥胖的症状都很好的效果，像冬瓜，西红柿这样高纤维的蔬菜要多吃一些，孩子不可以吃像蛋糕，夏威夷果这些高热量的食物，而且不要让孩子总是吃完就躺在床上不动，家长在治疗小儿肥胖期间如果孩子情况严重就要及时去医院在医生的指导下给孩子治疗。

营养保健科,小儿肥胖能吃该如何医治,女宝宝，目前2岁，近期，观察到，我家孩子越来越胖了，而且，吃起来好像也特别不节制，叫他运动也不愿意，请问：小儿肥胖能吃该如何医治。,当孩子患上肥胖症的时候家长可以增加孩子的运动量和控制他的饮食来改变症状，家长要监督孩子做一些有氧运动像慢跑，游泳等，要给孩子多吃一些像苹果，猕猴桃，胡萝卜等食物，一定要禁止孩子吃蛋糕，板栗这些高热量的食物，生活中不要让孩子

In [7]:
print(len(data_list))

84344


In [8]:
def prepare_message(data_list):
    '''
    格式样例：
    [
      {
        "id": "identity_0",
        "conversations": [
          {
            "from": "user",
            "value": "你好"
          },
          {
            "from": "assistant",
            "value": "我是一个语言模型，我叫通义千问。"
          }
        ]
      }
    ]
    '''
    new_list = []
    for i, data in enumerate(data_list):
        _id = f"identity_{i}"
        new_list.append(
        {
            "id": _id,
            "conversations": [
                {
                    "from": "user",
                    "value": data["input"]
                },
                {
                    "from": "assistant",
                    "value": data["output"]
                }
            ]
        })
    return new_list   
        

In [9]:
def replace_name(s):
    s = s.replace('<NAME>', '机器人小言')
    s = s.replace('<AUTHOR>', 'BlactCat_')
    return s

In [10]:
import json
import random

def load_self_cong_data(filename):
    data_list = []
    for d in json.load(open(filename, "r", encoding="utf-8")):
        d["instruction"] = replace_name(d["instruction"])
        d["output"] = replace_name(d["output"])
        data_list.append({
            "id": random.randint(10000, 100000),
            "conversations": [
                {
                    "from": "user",
                    "value": d["instruction"]
                },
                {
                    "from": "assistant",
                    "value": d["output"]
                }
            ]
        })
    return data_list

In [11]:
self_cong_data = load_self_cong_data("self_cognition.json")

In [12]:
print(len(self_cong_data))
print(self_cong_data[1])

80
{'id': 10076, 'conversations': [{'from': 'user', 'value': '你好'}, {'from': 'assistant', 'value': '您好，我是 机器人小言，一个由 BlactCat_ 打造的人工智能助手，请问有什么可以帮助您的吗？'}]}


In [13]:
import random
format_data_list = prepare_message(data_list)
random.shuffle(format_data_list)
format_data_list = self_cong_data + format_data_list
print(format_data_list[0])
print(format_data_list[100])
print(len(format_data_list))

{'id': 72686, 'conversations': [{'from': 'user', 'value': '你好'}, {'from': 'assistant', 'value': '您好，我是 机器人小言，一个由 BlactCat_ 开发的 AI 助手，很高兴认识您。请问我能为您做些什么？'}]}
{'id': 'identity_46373', 'conversations': [{'from': 'user', 'value': '上青少年宫陪儿子学特长，看到许多孩子在学武术，儿子看到了也想学，但我担心学这个特长会影响孩子的身高，想问一下会不会有影响呢？ '}, {'from': 'assistant', 'value': '这是没影响的，您还是用不着太过担心，像武术这个活动对身体都是很好的建议您在孩子的青春期还是可以适当的增强营养为好。特别是钙质的摄取，骨骼的生长是需要有大量的钙质，可以多喝牛奶，小虾等含钙量丰富是食物。可以吃加州熊牌钙片、斯强牌维生素钙片，但是钙片的消除效果是没那么好的。同时恰当的体育运动也是非常重要，适当的增强体育锻炼'}]}
84424


In [14]:
train_data = format_data_list
test_data = format_data_list[84000:]
print("train data size:", len(train_data))
print("test data size:", len(test_data))

train data size: 84424
test data size: 424


In [15]:
compute_dtype = getattr(torch, "float16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

In [16]:
model_path = "/data/coding/llm/qwen/Qwen-7B-Chat"

In [17]:
original_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    quantization_config=quant_config,
    trust_remote_code=True
)

The model is automatically converting to fp16 for faster inference. If you want to disable the automatic precision, please manually add bf16/fp16/fp32=True to "AutoModelForCausalLM.from_pretrained".
Try importing flash-attention for faster inference...
Loading checkpoint shards: 100%|██████████| 8/8 [00:11<00:00,  1.47s/it]


In [18]:
!nvidia-smi

Sun Jul  5 21:37:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-32GB           Off |   00000000:07:00.0 Off |                    0 |
| N/A   33C    P0             68W /  300W |    6329MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False, trust_remote_code=True, padding_side="right")
tokenizer.pad_token_id = tokenizer.eod_id

In [45]:
%%time

# response, history = original_model.chat(tokenizer, "你好，请介绍一下你自己", history=None)
# print(response)
# response, history = original_model.chat(tokenizer, "孩子积食了怎么办？", history=history) 
# print(response)
# response, history = original_model.chat(tokenizer, "孩子身上长疹子了是啥原因呢", history=history)
# print(response)


response, history = original_model.chat(tokenizer, "孩童中耳炎流黄水要如何治疗", history=history)
print(response)

您好，如果你的孩子被确诊为小儿中耳炎了，首先是要对孩子的耳朵实施消炎治疗，一般需要药物局部治疗，以消炎解毒为主，也可以给孩子服食抗生素药物，另外要是中耳炎引起了耳膜穿孔的情况，那么是可以选择手术医治的，家长平时还要留意给孩子正确做好日常护理工作，尽量避免诱发因素，祝宝宝早日康复。
CPU times: user 10.6 s, sys: 0 ns, total: 10.6 s
Wall time: 10.6 s


In [21]:
from transformers.trainer_pt_utils import LabelSmoother

IGNORE_TOKEN_ID = LabelSmoother.ignore_index

def preprocess(
    sources,
    tokenizer: AutoTokenizer,
    max_len: int,
    system_message: str = "You are a helpful assistant."
):
    
    roles = {"user": "<|im_start|>user", "assistant": "<|im_start|>assistant"}

    im_start = tokenizer.im_start_id
    im_end = tokenizer.im_end_id
    nl_tokens = tokenizer('\n').input_ids
    _system = tokenizer('system').input_ids + nl_tokens
    _user = tokenizer('user').input_ids + nl_tokens
    _assistant = tokenizer('assistant').input_ids + nl_tokens

    # Apply prompt templates
    input_ids, targets = [], []
    for i, source in enumerate(sources):
        if roles[source[0]["from"]] != roles["user"]:
            source = source[1:]

        input_id, target = [], []
        system = [im_start] + _system + tokenizer(system_message).input_ids + [im_end] + nl_tokens
        input_id += system
        target += [im_start] + [IGNORE_TOKEN_ID] * (len(system)-3) + [im_end] + nl_tokens
        assert len(input_id) == len(target)
        for j, sentence in enumerate(source):
            role = roles[sentence["from"]]
            _input_id = tokenizer(role).input_ids + nl_tokens + \
                tokenizer(sentence["value"]).input_ids + [im_end] + nl_tokens
            input_id += _input_id
            if role == '<|im_start|>user':
                _target = [im_start] + [IGNORE_TOKEN_ID] * (len(_input_id)-3) + [im_end] + nl_tokens
            elif role == '<|im_start|>assistant':
                _target = [im_start] + [IGNORE_TOKEN_ID] * len(tokenizer(role).input_ids) + \
                    _input_id[len(tokenizer(role).input_ids)+1:-2] + [im_end] + nl_tokens
            else:
                raise NotImplementedError
            target += _target
        assert len(input_id) == len(target)
        input_id += [tokenizer.pad_token_id] * (max_len - len(input_id))
        target += [IGNORE_TOKEN_ID] * (max_len - len(target))
        input_ids.append(input_id[:max_len])
        targets.append(target[:max_len])
    input_ids = torch.tensor(input_ids, dtype=torch.int)
    targets = torch.tensor(targets, dtype=torch.int)

    return dict(
        input_ids=input_ids,
        labels=targets,
        attention_mask=input_ids.ne(tokenizer.pad_token_id),
    )

In [22]:

class SupervisedDataset(Dataset):
    """Dataset for supervised fine-tuning."""

    def __init__(self, raw_data, tokenizer, max_len: int):
        super(SupervisedDataset, self).__init__()

        print("Formatting inputs...")
        sources = [example["conversations"] for example in raw_data]
        data_dict = preprocess(sources, tokenizer, max_len)

        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        self.attention_mask = data_dict["attention_mask"]
        print("Formatting done...")

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        return dict(
            input_ids=self.input_ids[i],
            labels=self.labels[i],
            attention_mask=self.attention_mask[i],
        )

In [23]:
train_dataset = SupervisedDataset(train_data[:1000], tokenizer, max_len=1024)
print(len(train_dataset))

Formatting inputs...
Formatting done...
1000


In [24]:
print(tokenizer)

QWenTokenizer(name_or_path='/data/coding/llm/qwen/Qwen-7B-Chat', vocab_size=151851, model_max_length=32768, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	
}


In [25]:
print(train_dataset[0])

{'input_ids': tensor([151644,   8948,    198,  ..., 151643, 151643, 151643],
       dtype=torch.int32), 'labels': tensor([151644,   -100,   -100,  ...,   -100,   -100,   -100],
       dtype=torch.int32), 'attention_mask': tensor([ True,  True,  True,  ..., False, False, False])}


In [26]:
test_dataset = SupervisedDataset(test_data, tokenizer, max_len=1024)

print(test_dataset[0])

Formatting inputs...
Formatting done...
{'input_ids': tensor([151644,   8948,    198,  ..., 151643, 151643, 151643],
       dtype=torch.int32), 'labels': tensor([151644,   -100,   -100,  ...,   -100,   -100,   -100],
       dtype=torch.int32), 'attention_mask': tensor([ True,  True,  True,  ..., False, False, False])}


In [27]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
config = LoraConfig(
    r=32, #Rank
    lora_alpha=16,
    target_modules=["c_attn", "c_proj", "w1", "w2"],
    bias="none",
    lora_dropout=0.05, # Conventional
    task_type="CAUSAL_LM",
)
# 1 - Enabling gradient checkpointing to reduce memory usage during fine-tuning
original_model.gradient_checkpointing_enable()
# 2 - Using the prepare_model_for_kbit_training method from PEFT
original_model = prepare_model_for_kbit_training(original_model)
peft_model = get_peft_model(original_model, config)

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.
You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


In [28]:
output_dir = './checkpoints_self_cong/'
import transformers
peft_training_args = TrainingArguments(
    output_dir = output_dir,
    warmup_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    logging_steps=100,
    logging_dir="./logs",
    save_strategy="steps",
    max_steps=1000,
    save_steps=100,
    evaluation_strategy="steps",
    eval_steps=100,
    do_eval=True,
    gradient_checkpointing=True,
    report_to="none",
    overwrite_output_dir = 'True',
    group_by_length=True,
)
peft_model.config.use_cache = False
peft_trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    args=peft_training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

/data/miniconda/envs/torch/lib/python3.10/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


In [29]:
# torch.cuda.empty_cache()
peft_trainer.train()

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


/data/miniconda/envs/torch/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss
100,2.729600,2.484807
200,2.305900,2.372631
300,2.192900,2.306319
400,1.989000,2.219647
500,2.176800,2.179896
600,2.067000,2.152015
700,2.012400,2.123440
800,2.045000,2.107776
900,1.957900,2.085481
1000,1.848600,2.079015


/data/miniconda/envs/torch/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/data/miniconda/envs/torch/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/data/miniconda/envs/torch/lib/python3.10/site-packages/to

TrainOutput(global_step=1000, training_loss=2.13252099609375, metrics={'train_runtime': 3503.0871, 'train_samples_per_second': 0.285, 'train_steps_per_second': 0.285, 'total_flos': 4.405592064e+16, 'train_loss': 2.13252099609375, 'epoch': 1.0})

In [37]:
# Free memory 
#del original_model
#del peft_trainer
torch.cuda.empty_cache()

In [31]:
!nvidia-smi

Sun Jul  5 22:35:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-32GB           Off |   00000000:07:00.0 Off |                    0 |
| N/A   48C    P0             66W /  300W |   27447MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [38]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
            
    return f"trainable model parameters: {trainable_model_params}\n \
        all model parameters: {all_model_params}\n \
        percentage of trainable model parameters:  \
        {100 * trainable_model_params / all_model_params:.2f}%"

In [39]:
print_number_of_trainable_model_parameters(original_model)

'trainable model parameters: 0\n         all model parameters: 4554887168\n         percentage of trainable model parameters:          0.00%'

In [40]:
compute_dtype = getattr(torch, "float16")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model_path = "/data/coding/llm/qwen/Qwen-7B-Chat"

original_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    quantization_config=quant_config,
    trust_remote_code=True
)

The model is automatically converting to fp16 for faster inference. If you want to disable the automatic precision, please manually add bf16/fp16/fp32=True to "AutoModelForCausalLM.from_pretrained".
Try importing flash-attention for faster inference...


Loading checkpoint shards: 100%|██████████| 8/8 [00:10<00:00,  1.35s/it]


In [41]:
from peft import PeftModel

output_dir = './checkpoints_self_cong/'

ft_model = PeftModel.from_pretrained(
    original_model, 
    output_dir + '/checkpoint-1000',
    torch_dtype=compute_dtype,
    device_map={"": 0},
    quantization_config=quant_config
)

In [ ]:
# response, history = ft_model.chat(tokenizer, "你好，请问你是由谁创造的？", history=None)
# print(response)

# print("--"*20)

# response, history = ft_model.chat(tokenizer, "孩子积食了怎么办？", history=None) 
# print(response)

# print("--"*20)

# response, history = ft_model.chat(tokenizer, "孩子身上长疹子了是啥原因呢", history=None)

# print(response)



# print("--"*20)


response, history = ft_model.chat(tokenizer, "怎么判断新生儿黄疸是病理性黄疸", history=None)

print(response)


response, history = original_model.chat(tokenizer, "怎么判断新生儿黄疸是病理性黄疸", history=history)
print(response)


“婴儿黄疸”是儿童常见症状之一。在临床上，除生理性和病理性黄疸外，还有许多混合性黄疸。而根据婴儿黄疸的性质和原因，可以分为多种类型。每种类型的婴儿黄疸都会给父母带来不同的心理压力，因此有必要及时治疗。建议去医院检查和确认结果，并在医生指导下进行药物治疗。
当孩子出现黄疸时，根据具体原因，可将其分为生理性和病理因素。对于新生儿黄疸，必须采取积极的预防措施。除了减少与药物和化学物质的接触之外，还应该确保摄入足够的维生素，尤其是胡萝卜素。此外，如果发现孩子黄疸并已形成后遗症，则需要立即就医。
